In [1]:
from dataclasses import dataclass
from time import perf_counter

import numpy as np

### 1.1. Функция и градиент

Для задачи 1 используем функцию и константы из ЛР1:

$$f(x)=\frac12\|Ax-b\|_2^2+\frac\alpha2\|x-a\|_2^2,$$

$$\nabla f(x)=A^\top(Ax-b)+\alpha(x-a),\qquad H=A^\top A+\alpha I_n,$$

$$L=\lambda_{\max}(H),\qquad \mu=\lambda_{\min}(H).$$

При $m<n$ имеем $\mu=\alpha$.

In [2]:
@dataclass
class Problem:
    A: np.ndarray
    a: np.ndarray
    b: np.ndarray
    alpha: float
    group: str = ""


def objective(problem, x):
    residual = problem.A @ x - problem.b
    displacement = x - problem.a
    return float(
        0.5 * (residual @ residual)
        + 0.5 * problem.alpha * (displacement @ displacement)
    )


def gradient(problem, x):
    return problem.A.T @ (problem.A @ x - problem.b) + problem.alpha * (x - problem.a)


def spectral_constants(problem):
    m, n = problem.A.shape
    singular_values = np.linalg.svd(problem.A, compute_uv=False)
    largest = float(singular_values[0])
    rank_tolerance = np.finfo(float).eps * max(m, n) * largest
    full_column_rank = m >= n and singular_values[-1] > rank_tolerance
    smallest_squared = float(singular_values[-1] ** 2) if full_column_rank else 0.0
    return largest ** 2 + problem.alpha, smallest_squared + problem.alpha

### 1.2. Метод Нестерова

Для пункта 1a используем метод Нестерова.
При $\mu>0$:

$$y_k=x_k+\beta(x_k-x_{k-1}),\qquad
x_{k+1}=y_k-\frac1L\nabla f(y_k),$$

$$\beta=\frac{\sqrt L-\sqrt\mu}{\sqrt L+\sqrt\mu},\qquad x_{-1}=x_0.$$

При $\mu=0$ используем переменный моментум, $y_0=x_0$, $t_0=1$:

$$x_{k+1}=y_k-\frac1L\nabla f(y_k),\qquad
t_{k+1}=\frac{1+\sqrt{1+4t_k^2}}2,$$

$$y_{k+1}=x_{k+1}+\frac{t_k-1}{t_{k+1}}(x_{k+1}-x_k).$$

In [3]:
def nesterov_steps(grad, x0, L, mu):
    x = np.array(x0, dtype=float, copy=True)
    y = x.copy()
    t = 1.0
    beta = (np.sqrt(L) - np.sqrt(mu)) / (np.sqrt(L) + np.sqrt(mu))

    while True:
        next_x = y - grad(y) / L
        if mu > 0:
            momentum = beta
        else:
            next_t = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * t * t))
            momentum = (t - 1.0) / next_t
            t = next_t
        y = next_x + momentum * (next_x - x)
        x = next_x
        yield x

### 1.3. Линейный каплинг

Для пункта 1b используем линейный каплинг.
Начальные значения: $y_0=z_0=x_0$. Выбираем коэффициенты

$$h=\frac1L,\qquad \tau_k=\frac2{k+2},\qquad \gamma_k=\frac{k+2}{2L}.$$

На итерации $k=0,1,\ldots$:

$$x_k=\tau_k z_k+(1-\tau_k)y_k,\qquad g_k=\nabla f(x_k),$$

$$y_{k+1}=x_k-hg_k,\qquad z_{k+1}=z_k-\gamma_k g_k.$$

Результат итерации - $y_{k+1}$. Один градиент используется в обоих
обновлениях. Формулы одинаковы при $\mu=0$ и $\mu>0$.

In [4]:
def linear_coupling_steps(grad, x0, L, mu):
    y = np.array(x0, dtype=float, copy=True)
    z = y.copy()
    k = 0

    while True:
        tau = 2.0 / (k + 2)
        gamma = (k + 2) / (2.0 * L)
        x = tau * z + (1.0 - tau) * y
        g = grad(x)
        y = x - g / L
        z = z - gamma * g
        k += 1
        yield y

### 1.4. Критерий остановки

Используем $f(x_k)-f^*\le\varepsilon$. Для заданного минимизатора $x^*$:

$$f(x)-f^*=\frac12\|A(x-x^*)\|_2^2+
\frac\alpha2\|x-x^*\|_2^2.$$


In [5]:
EPS = 0.01
MAX_ITER = 10_000


def objective_gap(problem, x, x_star):
    displacement = x - x_star
    residual = problem.A @ displacement
    return float(
        0.5 * (residual @ residual)
        + 0.5 * problem.alpha * (displacement @ displacement)
    )


def run_method(
    problem,
    x0,
    method,
    *,
    L,
    mu,
    x_star,
    tol=EPS,
    max_iter=MAX_ITER,
    grad=None,
):
    if grad is None:
        grad = lambda x: gradient(problem, x)

    x = np.array(x0, dtype=float, copy=True)
    steps = method(grad, x, L, mu)
    status = "max_iter"
    started = perf_counter()

    for iterations in range(max_iter + 1):
        gap = objective_gap(problem, x, x_star)
        if not np.isfinite(gap):
            status = "nonfinite"
            break
        if gap <= tol:
            status = "converged"
            break
        if iterations < max_iter:
            x = next(steps)

    return {
        "x": x,
        "iterations": iterations,
        "time": perf_counter() - started,
        "gap": gap,
        "status": status,
    }


methods = {
    "nesterov": nesterov_steps,
    "linear_coupling": linear_coupling_steps,
}